# Instationary control problems

In this notebook, we show how to solve an instationary linear control problem. The modules that one can employ for this type of control problem are similar to the one for the stationary case, with the exception that the callables have to take into account for the time variable. 

## Instationary convection–diffusion control problem

Given $\beta>0$, $\Omega \subset \mathrm{R}^d$, with $d \in \{1,2,3\}$, $\epsilon > 0$, and $t_f>0$, we consider the solution of the following instationary convection–diffusion control problem:

$$
    \min_{v, u} ~ \frac{1}{2} \int_0^{t_f} \| v - v_d
		\|^2 \mathrm{d} t + \frac{\beta}{2} \int_0^{t_f} 
		\| u \|^2 \mathrm{d} t
$$

subject to

$$
    \frac{\partial v}{\partial t} - \epsilon \nabla^2 v +
				\vec{w} \cdot \nabla v = f + u \qquad \mathrm{in} \;
					\Omega \times (0, t_f),
$$

provided with suitable initial and boundary conditions.

The module Control.Instationary allows the user to solve the previous problem by providing a few lines of code. For example, suppose we would like to solve the previous problem in $\Omega:=[-1,1]^2 \times [0, 2]$ (so that $\mathbf{x}=(x_1,x_2)$), starting from the initial condition

$$
    v(\mathbf{x},0)= 1 \qquad \mathrm{on} \; \partial \Omega_1 := \{1\} \times [-1,1],
$$
$$
    v(\mathbf{x},0)= 0 \qquad \mathrm{on} \; \partial \Omega_2 := \partial \Omega \setminus \partial \Omega_1,
$$

and with boundary conditions given by

$$
    v(\mathbf{x},t)= 1 \qquad \mathrm{on} \; \partial \Omega_1 \times [0,2],
$$
$$
    v(\mathbf{x},t)= 0 \qquad \mathrm{on} \; \partial \Omega_2 \times [0,2].
$$

For this example, we consider the wind $\vec{w}(\mathbf{x},t) = \sin(\pi t)[2 x_2 (1-x_1^2), -2 x_1 (1- x_2^2)]^\top$, with diffusion parameter $\epsilon = \frac{1}{250}$ and regularization parameter $\beta = 10^{-2}$. Further, we set $f=0$, and seek the desired state $v_d = e^{-10(1-x_1)}$.

After importing the important modules, we first define the function space where to seek the solution (in this example, we employ $P_1$ finite elements), then we define the force function, the desired state, the bilinear form, and the initial and boundary conditions. Note that, since this is a time-dependent problem, the callables accept as input the time $t$ where to evaluate the functions.

In [ ]:
from firedrake import *
from control.control import *

beta = 1.0e-2

mesh = RectangleMesh(10, 10, 1.0, 1.0, originX=-1.0, originY=-1.0)

space_0 = FunctionSpace(mesh, "Lagrange", 1)


# the force funciton
def force_f(test, t):
    space = test.function_space()

    # force function
    f = Function(space)
    f.zero()

    return inner(f, test) * dx


# the desired state
def desired_state(test, t):
    space = test.function_space()
    mesh = space.mesh()
    X = SpatialCoordinate(mesh)

    # desired state
    v_d = Function(space, name="v_d")
    v_d.interpolate(exp(- 10.0 * (1.0 - X[0])))

    return inner(v_d, test) * dx, v_d


# the forward form
def forw_diff_operator(trial, test, u, t):
    space = test.function_space()
    mesh = space.mesh()
    X = SpatialCoordinate(mesh)
    x_1 = X[0]
    x_2 = X[1]

    # diffusion parameter
    epsilon = 1.0 / 250.0

    # wind
    w = as_vector([sin(pi * t) * 2.0 * x_2 * (1.0 - x_1 * x_1),
                   -sin(pi * t) * 2.0 * x_1 * (1.0 - x_2 * x_2)])

    # spatial differential for the forward problem
    return (
        epsilon * inner(grad(trial), grad(test)) * dx
        + inner(dot(grad(trial), w), test) * dx)


# auxiliary boundary conditions
bcs_v = [DirichletBC(space_0, Constant(1.0), (4,)),
         DirichletBC(space_0, 0.0, (1, 2, 3))]


# the initial condition
def initial_condition(test):
    space = test.function_space()

    v_0 = Function(space)
    v_0.zero()

    for bc in bcs_v:
        bc.apply(v_0)

    return v_0


# the boundary conditions
def bcs_v_t(space_0, t):
    return bcs_v

We can now define the problem by calling the constructor Control.Instationary. This is done by passing all the previous callable to it, together with the time interval for integration and the number of points $n_t$ to employ in time (for this example, we will employ $n_t=20$; the default option is $n_t=10$). Note that one can also privde to the constructor the type of discretization employed by passing the kwarg CN. For example, suppose one would like to employ backward Euler as time discretization; then, one has to pass CN=False (the default option is CN=True). We mention that one can switch from backward Euler to trapezi in time by calling the module set_CN(), that accepts the kwarg CN (the default option is CN=True). Further, we note that, for this example, the linear system to be solver for is not symmetric when applying a trapezoidal rule in time.

In [ ]:
n_t = 20

instationary_convection_diffusion_control = Control.Instationary(
    space_0, forw_diff_operator, desired_state=desired_state,
    force_function=force_f, beta=beta, CN=False, n_t=n_t,
    initial_condition=initial_condition,
    time_interval=(0.0, 2.0), bcs_v=bcs_v_t)

We can now call the linear solver with the module linear_solve(). This module accepts the same input as the linear_solve() module in the Stationary class, so one ca modify the linear solver by passing the appropriate kwargs.

In [ ]:
instationary_convection_diffusion_control.linear_solve(
    print_error=False, create_output=False, plots=False)

In case one wishes to solve a non-linear instationary problem, one has to call the module non_linear_solve(). The kwarg and default options are the same as for the non_linear_solve() of the Stationary class.